# Glove gesture classifier

Loads a `.keras` model trained by `1D_CNN_variants_lstm_generalisation.ipynb`
and classifies every CSV under a folder of the same structure
(`<user>/<gesture_label>/*.csv`).

The preprocessing pipeline mirrors the training notebook **exactly**:

1. Trim each CSV to the first `USE_SECONDS` of the recording
2. Linear resample to `RESAMPLE_TO_N_STEPS`
3. (Optional) zero-phase Butterworth low-pass filter
4. Per-trial normalisation (z-score / min-max / none)
5. Apply the fitted scaler (if one was saved alongside the model)
6. Predict, then compare with the ground-truth folder label

Outputs:

- Overall accuracy + per-class precision / recall / F1
- Per-trial CSV with `(file, true_label, predicted_label, max_prob)`
- Confusion matrix figure

## 1. Configuration

Set model + data paths here. Everything else is derived.

In [ ]:
from pathlib import Path

# ── Required: artefacts produced by the training notebook ────────────────────
MODEL_PATH  = Path('/home/jestin/ThesisRepo/ML/SavedModels/cnn_BN_GAP_Wide_YYYYMMDD_HHMMSS.keras')
LABELS_PATH = Path('/home/jestin/ThesisRepo/ML/SavedModels/labels_BN_GAP_Wide_YYYYMMDD_HHMMSS.json')
SCALER_PATH = Path('/home/jestin/ThesisRepo/ML/SavedModels/scaler_BN_GAP_Wide_YYYYMMDD_HHMMSS.joblib')
#   Set SCALER_PATH = None if the training run used PER_TRIAL_NORM != 'none'
#   (in that case no scaler was saved and the labels.json filename will be the
#   only sibling artefact).

# ── Required: folder to classify ─────────────────────────────────────────────
#   Expected layout (identical to the training notebook):
#     DATA_ROOT/<user>/<gesture_label>/*.csv
#   Set USERS = None to classify every user folder under DATA_ROOT.
DATA_ROOT = Path('/home/jestin/ThesisRepo/ML/NewTestData')
USERS     = None     # e.g. ['18_Bridgette', '15_NewUser']  →  classify just these

# ── Sensor columns (must match training) ─────────────────────────────────────
SENSOR_COLS = [
    'flex0', 'flex1', 'flex2', 'flex3', 'flex4',
    'imu0_ax', 'imu0_ay', 'imu0_az', 'imu0_gx', 'imu0_gy', 'imu0_gz',
    'imu1_ax', 'imu1_ay', 'imu1_az', 'imu1_gx', 'imu1_gy', 'imu1_gz',
    'imu2_ax', 'imu2_ay', 'imu2_az', 'imu2_gx', 'imu2_gy', 'imu2_gz',
]

# ── Preprocessing knobs (MUST match the training run) ────────────────────────
# These should mirror the values that were active in the training notebook
# when MODEL_PATH was saved. The experiment_log.csv row for that timestamp
# holds the exact values — copy them across.
SAMPLING_RATE_HZ       = 30.0
GESTURE_DURATION_S     = 3.0
USE_SECONDS            = 3.0       # seconds from start of each trial to keep
APPLY_BUTTERWORTH      = True
BUTTERWORTH_CUTOFF_HZ  = 6.0
BUTTERWORTH_ORDER      = 4
PER_TRIAL_NORM         = 'zscore'                    # 'zscore' | 'minmax' | 'none'
PER_TRIAL_MINMAX_RANGE = (0.0, 1.0)
EXCLUDE_CLASSES        = set()     # any class folder names you want to skip

# ── Output ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path('/home/jestin/ThesisRepo/ML/ClassifierOutputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Derived
USE_SECONDS    = float(min(max(USE_SECONDS, 0.0), GESTURE_DURATION_S))
_USE_N_SAMPLES = max(2, int(round(USE_SECONDS * SAMPLING_RATE_HZ)))
RESAMPLE_TO_N_STEPS = _USE_N_SAMPLES
print(f'Using first {USE_SECONDS:.2f} s ({_USE_N_SAMPLES} samples) of each trial')
print(f'Per-trial normalisation : {PER_TRIAL_NORM}')
print(f'Butterworth             : {APPLY_BUTTERWORTH} '
      f'(cutoff {BUTTERWORTH_CUTOFF_HZ} Hz, order {BUTTERWORTH_ORDER})')


## 2. Imports and preprocessing helpers

These functions are byte-for-byte copies of the training notebook so the trial preprocessing is identical.

In [ ]:
import json, glob, os
import numpy as np
import pandas as pd
import joblib
from scipy import signal as scipy_signal
import tensorflow as tf
from tensorflow import keras


def resample_trial(trial, n_steps):
    '''Linear resample (T, C) -> (n_steps, C). Mirrors training notebook.'''
    T, C = trial.shape
    if T == n_steps:
        return trial.astype(np.float32, copy=False)
    old_idx = np.linspace(0, 1, T)
    new_idx = np.linspace(0, 1, n_steps)
    out = np.zeros((n_steps, C), dtype=np.float32)
    for c in range(C):
        out[:, c] = np.interp(new_idx, old_idx, trial[:, c])
    return out


def apply_butterworth(trials, cutoff, order, fs):
    '''Zero-phase low-pass filter per channel. Mirrors training notebook.'''
    nyq = fs / 2.0
    norm_cutoff = cutoff / nyq
    if norm_cutoff >= 1.0:
        print(f'  WARNING: cutoff {cutoff} Hz >= Nyquist {nyq} Hz — skipping filter.')
        return trials
    b, a = scipy_signal.butter(order, norm_cutoff, btype='low', analog=False)
    return [scipy_signal.filtfilt(b, a, t, axis=0).astype(np.float32) for t in trials]


def per_trial_zscore(trials):
    out = []
    for t in trials:
        mu = t.mean(axis=0, keepdims=True)
        sd = t.std(axis=0, keepdims=True)
        sd = np.where(sd < 1e-8, 1.0, sd)
        out.append(((t - mu) / sd).astype(np.float32))
    return out


def per_trial_minmax(trials, out_range=(0.0, 1.0)):
    lo, hi = out_range
    out = []
    for t in trials:
        mn = t.min(axis=0, keepdims=True)
        mx = t.max(axis=0, keepdims=True)
        rng = np.where((mx - mn) < 1e-8, 1.0, (mx - mn))
        out.append((((t - mn) / rng) * (hi - lo) + lo).astype(np.float32))
    return out


def per_trial_normalise(trials, mode, minmax_range=(0.0, 1.0)):
    if mode == 'zscore':
        return per_trial_zscore(trials)
    if mode == 'minmax':
        return per_trial_minmax(trials, out_range=minmax_range)
    if mode == 'none':
        return trials
    raise ValueError(f"PER_TRIAL_NORM must be 'zscore', 'minmax' or 'none', got {mode!r}")


## 3. Load the trained model

In [ ]:
model = keras.models.load_model(MODEL_PATH)
print(f'Loaded model from {MODEL_PATH}')
print(f'Input shape  : {model.input_shape}   (None, T={model.input_shape[1]}, C={model.input_shape[2]})')
print(f'Output shape : {model.output_shape}')

# Sanity-check sequence length against the trim/resample config
_, T_expected, C_expected = model.input_shape
if T_expected != RESAMPLE_TO_N_STEPS:
    print(f'\n  WARNING: model expects T={T_expected} but USE_SECONDS={USE_SECONDS} '
          f'gives {RESAMPLE_TO_N_STEPS} samples.')
    print(f'  Adjust USE_SECONDS in Section 1 to match (T_expected/SAMPLING_RATE_HZ '
          f'= {T_expected/SAMPLING_RATE_HZ:.2f} s).')
if C_expected != len(SENSOR_COLS):
    print(f'\n  WARNING: model expects C={C_expected} channels but '
          f'SENSOR_COLS has {len(SENSOR_COLS)}.')

class_names = json.loads(LABELS_PATH.read_text())
print(f'\n{len(class_names)} class labels: {class_names}')

# Scaler is optional — only present when PER_TRIAL_NORM == 'none'
scaler = None
if SCALER_PATH is not None and SCALER_PATH.exists():
    scaler = joblib.load(SCALER_PATH)
    print(f'Loaded scaler from {SCALER_PATH}  ({type(scaler).__name__})')
else:
    print('No scaler loaded (per-trial normalisation was used at training time).')


## 4. Load and preprocess every CSV under `DATA_ROOT`

In [ ]:
def _list_users(root, users=None):
    if users is None:
        return sorted([d.name for d in root.iterdir() if d.is_dir()])
    return [u for u in users if (root / u).is_dir()]


def _list_dynamic_dir(user_dir):
    '''Mirror training-notebook expectation: <user>/Dynamic/<label>/*.csv.
    Falls back to <user>/<label>/*.csv if there's no Dynamic subfolder.'''
    dyn = user_dir / 'Dynamic'
    return dyn if dyn.is_dir() else user_dir


def load_user(user_dir):
    '''Return (X_raw_arr, labels, files) for one user.

    X_raw_arr is shape (n_trials, RESAMPLE_TO_N_STEPS, len(SENSOR_COLS)),
    preprocessed up to and including per-trial normalisation (matches
    training-time per_user_data values).
    '''
    base = _list_dynamic_dir(user_dir)
    trials, labels, files = [], [], []
    for label_dir in sorted(p for p in base.iterdir() if p.is_dir()):
        label = label_dir.name
        if label in EXCLUDE_CLASSES:
            continue
        for fp in sorted(label_dir.glob('*.csv')):
            try:
                df = pd.read_csv(fp)
                avail = [c for c in SENSOR_COLS if c in df.columns]
                if len(avail) != len(SENSOR_COLS):
                    missing = set(SENSOR_COLS) - set(df.columns)
                    print(f'  SKIP {fp}: missing channels {sorted(missing)}')
                    continue
                arr = df[SENSOR_COLS].values.astype(np.float32)
                if _USE_N_SAMPLES < arr.shape[0]:
                    arr = arr[:_USE_N_SAMPLES]
                trials.append(arr)
                labels.append(label)
                files.append(str(fp))
            except Exception as e:
                print(f'  ERR {fp}: {e}')
    if not trials:
        return None, [], []
    trials = [resample_trial(t, RESAMPLE_TO_N_STEPS) for t in trials]
    if APPLY_BUTTERWORTH:
        trials = apply_butterworth(trials, BUTTERWORTH_CUTOFF_HZ,
                                   BUTTERWORTH_ORDER, SAMPLING_RATE_HZ)
    trials = per_trial_normalise(trials, PER_TRIAL_NORM, PER_TRIAL_MINMAX_RANGE)
    X = np.stack(trials, axis=0).astype(np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X, labels, files


all_X, all_y_str, all_files, all_users = [], [], [], []
for u in _list_users(DATA_ROOT, USERS):
    print(f'\nLoading {u}...')
    Xu, yu, fu = load_user(DATA_ROOT / u)
    if Xu is None:
        print(f'  no trials found')
        continue
    print(f'  {Xu.shape[0]} trials, shape {Xu.shape}')
    all_X.append(Xu)
    all_y_str.extend(yu)
    all_files.extend(fu)
    all_users.extend([u] * Xu.shape[0])

assert all_X, 'No trials loaded — check DATA_ROOT and folder structure.'
X = np.concatenate(all_X, axis=0)
print(f'\nTotal: {X.shape[0]} trials across {len(set(all_users))} users')
print(f'X shape: {X.shape}')


## 5. Apply the scaler (if any) and predict

In [ ]:
if scaler is not None:
    Nt, T, C = X.shape
    X_s = scaler.transform(X.reshape(Nt, T * C)).reshape(Nt, T, C).astype(np.float32)
    print(f'Applied scaler ({type(scaler).__name__})')
else:
    X_s = X

probs = model.predict(X_s, verbose=1)
y_pred_idx = np.argmax(probs, axis=1)
y_pred_str = [class_names[i] for i in y_pred_idx]
max_prob   = probs.max(axis=1)
print(f'\nPredicted {len(y_pred_idx)} trials.')


## 6. Evaluate against ground-truth folder labels

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)
import matplotlib.pyplot as plt
from datetime import datetime

# Filter to only trials whose true label is in the model's class set.
# Anything outside class_names is reported separately ('out-of-vocab').
in_vocab = np.array([y in class_names for y in all_y_str])
oov_count = int((~in_vocab).sum())
if oov_count:
    print(f'  {oov_count} trials have labels not seen at training time '
          f'(treated as classification errors).')

y_true_str = np.asarray(all_y_str)
acc_overall = accuracy_score(y_true_str[in_vocab], np.asarray(y_pred_str)[in_vocab]) if in_vocab.any() else float('nan')
print(f'\nOverall accuracy (in-vocab only): {acc_overall:.4f}')

# Per-class report
print('\nClassification report:')
print(classification_report(y_true_str[in_vocab], np.asarray(y_pred_str)[in_vocab],
                            labels=class_names, zero_division=0))

# Per-user breakdown
print('Per-user accuracy:')
all_users_arr = np.asarray(all_users)
for u in sorted(set(all_users_arr)):
    m = (all_users_arr == u) & in_vocab
    if not m.any():
        continue
    a = accuracy_score(y_true_str[m], np.asarray(y_pred_str)[m])
    print(f'  {u:20s}  n={int(m.sum()):3d}  acc={a:.4f}')

# Confusion matrix figure
cm = confusion_matrix(y_true_str[in_vocab], np.asarray(y_pred_str)[in_vocab],
                      labels=class_names)
fig, ax = plt.subplots(figsize=(1.4 + 0.6*len(class_names), 1.4 + 0.6*len(class_names)))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion matrix — acc {acc_overall:.3f} (n={int(in_vocab.sum())})')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black',
                fontsize=9)
fig.colorbar(im, fraction=0.046, pad=0.04)
fig.tight_layout()

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
cm_path = OUTPUT_DIR / f'classifier_confusion_{ts}.png'
fig.savefig(cm_path, dpi=160, bbox_inches='tight')
print(f'\nConfusion matrix saved to {cm_path}')

# Per-trial CSV
trial_df = pd.DataFrame({
    'file':       all_files,
    'user':       all_users,
    'true_label': all_y_str,
    'predicted':  y_pred_str,
    'max_prob':   max_prob,
    'correct':    [t == p for t, p in zip(all_y_str, y_pred_str)],
})
trial_csv = OUTPUT_DIR / f'classifier_predictions_{ts}.csv'
trial_df.to_csv(trial_csv, index=False)
print(f'Per-trial predictions saved to {trial_csv}')


## 7. (Optional) Inspect the lowest-confidence and misclassified trials

In [ ]:
wrong = trial_df[trial_df['correct'] == False].sort_values('max_prob')
print(f'{len(wrong)} misclassified trials. Lowest-confidence examples:')
print(wrong.head(20).to_string(index=False))

print('\nLowest-confidence trials overall (any correctness):')
print(trial_df.sort_values('max_prob').head(15).to_string(index=False))
